In [ ]:
import pandas as pd
import TrajectoryPreprocessor as pp

# TrajectoryPreprocessor Class Tutorial

In [ ]:
raw_data = pd.read_csv('10_users_all_obs_raw.csv')

# raw_data = pd.read_csv('seattle_2000_all_obs_preprocessed_sampled.csv')
# raw_data['datetime'] = pd.to_datetime(raw_data['datetime']).astype('int64') // 10**6
# raw_data = raw_data.rename(columns={'UID': 'user_ID',
#                                     'datetime': 'unix_start_t'})
# raw_data = raw_data[['user_ID', 'unix_start_t', 'orig_lat', 'orig_long', 'orig_unc']]

raw_data

In [ ]:
# How many unique users are there?
unique_users = raw_data['user_ID'].nunique()
print(f'Number of unique users: {unique_users}')

In [ ]:
# Initialize TrajectoryPreprocessor
tp = pp.TrajectoryPreprocessor(raw_data)

This 'tp' object now contains the following attributes:
- `tp.raw_data`: the raw data that was passed to the preprocessor
- `tp.processed_data`: currently this is the same as the raw data, but it will be updated as the preprocessor runs different methods.
- `tp.col_maps`: a dictionary that maps the original column names to a set of new column names.

**Note that all methods within 'tp' consider one user at atime using the unique user_id. This is because the data is assumed to be sorted by user_id and timestamp.**

In [ ]:
tp.raw_data.head() # same as above!

We can now add some new variables that will be useful. Each time we run a method, we will update the `processed_data` attribute so we do not need to pass the data to each method. 

In [ ]:
tp.convert_timestamps() # convert timestamps to datetime objects

tp.processed_data.head() # Note the new 'datetime' column

We have now added a datetime mapping to the preprocessor object.

## Noise filtering
Specifically, we will remove any data points where the average speed since the last data point is greater than 200 km/h. This helps us get rid of erroneous data points that are likely due to GPS errors.

In [ ]:
tp.filter_by_speed(speed_threshold=200)

tp.processed_data.head()

In [ ]:
# How many rows were removed?
rows_removed = tp.raw_data.shape[0] - tp.processed_data.shape[0]
print(f'We filtered {rows_removed}' + ' GPS points by speed')

## Stay clustering
We will create and store a separate (much smaller) dataframe where each row denotes an activity at a fixed location. The parameters for this method are explained in the .py file.

In [ ]:
tp.stay_location_clustering(
    cluster_radius_km =0.2, 
    min_samples = 1,
    minutes_for_a_stop = 15,
    spatial_radius_km = 0.3, 
    leaving_time=True)

In [ ]:
tp.clustered_locations # Note that this is a different dataframe than preprocessor.processed_data (this only shows the stay locations)

Let's now associate each of the original data points with a cluster number (if they fall within one in 'preprocessor.clustered_locations') or -99 (if they are a trip).

In [ ]:
tp.merge_stay_clusters()

See our newest 'cluster' column in the data. These are unique to the user (i.e., the same cluster number may be used for different users, but this does not denote the same location).

In [ ]:
tp.processed_data.head()

Now let's add a unique trajectory identifier. The logic here is simple: a unique trajectory ID should capture the last point before a trip starts, the trip points, and the point where the trip ends. 
- So a cluster sequence like $[1, -99, -99, -99, -99, 2]$ would warrant one unique trajectory ID

In [ ]:
tp.assign_trajectory_ids()

Take a look at the last column!

In [ ]:
tp.processed_data.head()

# Trip-level Statistics
Let's now calculate some trip-level statistics. To do so, we need to first get the distance and vleoicty between each point. We will then calculate the following statistics:
- Trip distance
- Trip duration
- Start and end clusters
- Start and end times
- Average speed
- Maximum speed
- Number of points in the trip

In [ ]:
tp.add_dist_and_vel()

In [ ]:
tp.processed_data.head()

In [ ]:
tp.add_start_end_stay_clusters()

In [ ]:
tp.processed_data.head(100)

In [ ]:
tp.add_start_end_datetimes()

In [ ]:
tp.processed_data.head(100)

In [ ]:
tp.add_trip_metrics()

In [ ]:
tp.add_point_counts()

In [ ]:
tp.processed_data.head(100)

In [ ]:
tp.processed_data.to_csv('outputs/10_users_processed.csv', index=True, header=True, mode='w')
# tp.processed_data.to_csv('outputs/seattle_2000_processed.csv', index=True, header=True, mode='w')

# Compression
We can now create a much smaller dataframe where each row denotes one whole trip and summarizes the trip-level statistics. This is useful for further analysis.

In [ ]:
tp.create_compressed_trips()

In [ ]:
tp.compressed_trips

In [ ]:
tp.compressed_trips.to_csv('outputs/10_users_processed.csv', index=True, header=True, mode='w')
# tp.compressed_trips.to_csv('outputs/seattle_2000_processed.csv', index=True, header=True, mode='w')

**NOTE**: All methods within 'tp' are chainable, meaning that you can call them one after the other. For example:

In [ ]:
tp = pp.TrajectoryPreprocessor(raw_data)

tp.convert_timestamps()\
    .filter_by_speed(speed_threshold=200)\
        .stay_location_clustering()\
            .merge_stay_clusters()\
                .assign_trajectory_ids()\
                    .add_dist_and_vel()\
                        .add_start_end_stay_clusters()\
                            .add_start_end_datetimes()\
                                .add_trip_metrics()\
                                    .add_point_counts()\
                                        .create_compressed_trips()

The order that these methods are called is important, but I have not yet tested whether anything breaks if you call them in a different order. Feel free to do so and let me know if you encounter any issues!